In [ ]:
%load_ext autoreload
%autoreload 2

# modal

> Abstractions for creating a Modal Sandbox

In [ ]:
#| default_exp modal

In [ ]:
#| export
import modal

In [ ]:
#| export
def mk_app(name): return modal.App.lookup(name, create_if_missing=True)

In [ ]:
app = mk_app('solveit-sandbox'); app

In [ ]:
#| export
_default_condas = ['ipykernel', 'fastai', 'cuda-toolkit',]

In [ ]:
#| export
_default_pips = [
    'ipykernel', 'fastai',
    'transformers', 'diffusers', 'accelerate', 'datasets',
    'huggingface_hub', 'peft', 'bitsandbytes', 'safetensors', 'einops',
    'xformers', 'ninja', 'sentence-transformers', 'wandb', 'gradio',
    'python-fasthtml', 'plotly', 'fsspec', 's3fs', 'gcsfs', 'librosa',
    'imageio', 'imageio-ffmpeg', 'ipywidgets',
]

In [ ]:
#| export
def mk_img(
    condas=None,  # conda packages (defaults to _default_condas)
    pips=None,    # pip packages (defaults to _default_pips)
):
    if condas is None: condas = _default_condas
    if pips is None: pips = _default_pips
    # return (modal.Image.micromamba()
        # .apt_install('openssh-server')
        # .micromamba_install(condas)
        # .pip_install(pips)
    # )
    return modal.Image.debian_slim().apt_install('openssh-server').pip_install(pips)


In [ ]:
img = mk_img(); img

Image(<function _Image.pip_install.<locals>.build_dockerfile at 0x789e483bf2e0>)

In [ ]:
#| export
def get_apts() -> list[str]:  # installed system package names
    return run('dpkg-query -W -f=\'${Package}\\n\'').splitlines()

In [ ]:
get_apts()[:10]

['adduser',
 'alsa-topology-conf',
 'alsa-ucm-conf',
 'apt',
 'aria2',
 'autoconf',
 'automake',
 'autotools-dev',
 'base-files',
 'base-passwd']

In [ ]:
#| export
def get_pips() -> list[str]:  # installed python packages (name==version)
    return [l for l in run('pip freeze').splitlines() if not l.startswith(('-e', '@'))]

In [ ]:
get_pips()[:10]

['accelerate==1.13.0',
 'anki==25.9.2',
 'beartype==0.22.5',
 'blis==1.3.0',
 'catalogue==2.0.10',
 'cbor2==5.8.0',
 'cloudpathlib==0.23.0',
 'cloudpickle==3.1.2',
 'confection==0.1.5',
 'coolname==2.2.0']

In [ ]:
#| export
import json

In [ ]:
#| export
def get_secrets() -> dict:  # secrets from solveit_settings.json
    with open('/app/data/solveit_settings.json') as f: return json.load(f)['secrets']

In [ ]:
#| export
def mk_sandbox(
    app,            # modal.App to register the Sandbox with
    img,            # modal.Image for the Sandbox environment
    timeout=600,    # auto-terminate after this many seconds
    gpu=None,       # GPU type string (e.g. "T4", "A100") or None for no GPU
    secrets=None,   # dict of secrets (defaults to local solveit secrets)
):
    if secrets is None: secrets = get_secrets()
    return modal.Sandbox.create('sleep', 'infinity',
        app=app, image=img, timeout=timeout,
        unencrypted_ports=[22], gpu=gpu,
        secrets=[modal.Secret.from_dict(secrets)])


In [ ]:
sb = mk_sandbox(app, img, gpu='T4'); sb

Sandbox()

In [ ]:
#| export
def get_tunnel(sb):
    t = sb.tunnels()[22]
    return t.unencrypted_host, t.unencrypted_port

In [ ]:
host,port = get_tunnel(sb); host,port

('r433.modal.host', 40039)

In [ ]:
#| export
import os

In [ ]:
#| export
def get_pubkey(): return os.environ['SSH_PUBKEY']

In [ ]:
#| export
def inject_pubkey(sb, pubkey):
    sb.exec('mkdir', '-p', '/root/.ssh')
    sb.exec('bash', '-c', f'echo {pubkey} > /root/.ssh/authorized_keys')

In [ ]:
inject_pubkey(sb, get_pubkey())

In [ ]:
#| export
from time import sleep

In [ ]:
#| export
def start_ssh(sb):
    sb.exec('service', 'ssh', 'start')
    for i in range(20):
        proc = sb.exec('service', 'ssh', 'status')
        if 'fail' in proc.stdout.read(): sleep(0.3)
        elif 'run' in proc.stdout.read(): 
            print('✔ started ssh service')
            return
    print('✖ failed to start ssh service')

In [ ]:
start_ssh(sb)

✔ started ssh service


In [ ]:
#| export
from fastcore.all import *

**Why this change**: micromamba images keep Python outside the default `PATH`
(`~/.local/share/mamba/envs/...`), so bare `ssh` commands can't find `python3`, `pip`, etc.
Rather than forcing every call through `micromamba run` or login shells, `mk_ssh` now:

1. Figures out where Python actually lives on the remote machine
2. Prepends that `bin/` directory to `PATH` before every command

After this, `ssh('python3 -c "print(1+1)"')` and `ssh('pip list')` just work — no
`micromamba run` prefix needed.

This is equivalent to what `micromamba run` does internally, but baked into the
`ssh` helper so it's transparent to callers.

In [ ]:
#| export
def mk_ssh(host, port):
    return lambda *cmd: run('ssh', '-p', str(port),
        '-o', 'StrictHostKeyChecking=no',
        '-o', 'UserKnownHostsFile=~/.ssh/known_hosts',
        '-o', 'ConnectTimeout=10',
        f'root@{host}', *cmd)

In [ ]:
ssh = mk_ssh(host,port); ssh

<function __main__.mk_ssh.<locals>.<lambda>(*cmd)>

In [ ]:
#| export
def verify_ssh(ssh):
    "Verify SSH connection to Sandbox and print system info."
    h, u, w = ssh('hostname; uname -srmo; whoami').splitlines()[:3]
    sys_name, ver, arch, os_name = u.split()
    print(f'系统：  {sys_name}')
    print(f'主机名：{h}')
    print(f'用户：  {w}')
    print(f'内核：  {ver}')
    print(f'架构：  {arch}')
    print(f'系统类型：{os_name}')

In [ ]:
verify_ssh(ssh)

系统：  Linux
主机名：modal
用户：  root
内核：  4.4.0
架构：  x86_64
系统类型：GNU/Linux


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()